# Mapping the Ideological Subordination of Chinese Governance

> Does official CCP discourse treat governance as a politically neutral concept, or is it
> fundamentally subordinated to regime security?

In [ ]:
import numpy as np
import pandas as pd

from xi_discourse import config, lexicon
from xi_discourse.preprocess import build_corpus
from xi_discourse.train import load_all

## Corpus

See `xi_discourse.preprocess`.

Strips Xinhua bylines, removes duplicated sentences the scraper duplicated within documents, splits into sentences *before* stripping punctuation, and segments with a custom 'jieba' dictionary that keeps 一带一路, 国家安全, 党的领导 and
党 intact.

In [ ]:
sentences, meta, stats = build_corpus()

print(f"Documents analysed : {stats.documents_used:,}  (of {stats.documents_raw:,} scraped)")
print(f"Sentences          : {stats.sentences:,}")
print(f"Tokens analysed    : {stats.tokens:,}")
print(f"Mean sentence      : {stats.mean_sentence_len:.1f} tokens")
print(f"Coverage           : {stats.date_min} to {stats.date_max}")

In [ ]:
models = load_all()
primary = models[config.PRIMARY_SEED]
wv = primary.wv

print(f"Seeds loaded    : {sorted(models)}")
print(f"Vocabulary size : {len(wv):,}")

In [ ]:
rows = []
for field, terms in {**lexicon.ALL_SETS, "bri": lexicon.BRI_TERMS}.items():
    for zh, en in terms.items():
        rows.append({
            "field": field, "term": zh, "gloss": en,
            "in_vocab": zh in wv,
            "count": int(wv.get_vecattr(zh, "count")) if zh in wv else 0,
        })
coverage = pd.DataFrame(rows)
missing = coverage.loc[~coverage.in_vocab, "term"].tolist()
print("Missing from vocabulary:", missing or "none")
coverage.sort_values("count", ascending=False).head(20)

In [ ]:
from xi_discourse.stats import frequency_stratified_null, null_distribution

tech = lexicon.present(lexicon.TECHNOCRATIC, wv)
polg = lexicon.present(lexicon.POLITICAL_GOVERNANCE, wv)
ideo = lexicon.present(lexicon.IDEOLOGICAL, wv)
secu = lexicon.present(lexicon.REGIME_SECURITY, wv)
ecol = lexicon.present(lexicon.ECOLOGICAL, wv)

n_vocab = null_distribution(wv, "vocab")
n_freq = null_distribution(wv, "frequency")
n_strat = frequency_stratified_null(wv, tech + polg + ideo + secu)

for n in (n_vocab, n_freq, n_strat):
    print(f"{n.kind:22s} mean={n.mean:.3f}  sd={n.sd:.3f}  95th={n.p95:.3f}  n={n.n:,}")

## Pairwise cosine similarity

Party-doctrinal vocabulary is separated from regime-security vocabulary. The claim of administrative terms clustering
   around *regime security* is a claim about the second set only.

Political-governance compounds are dominated by *environmental remediation*
   sense: 66% of its nearest neighbours are ecological terms (风沙源 sandstorm source,
   水沙 water-sediment, 碳汇 carbon sink, 排污 effluent) against a 5% frequency-matched
   baseline

In [ ]:
from xi_discourse.stats import bootstrap_ci, cross_set_similarities, intra_set_similarities

comparisons = {
    "technocratic <-> ideological":              cross_set_similarities(wv, tech, ideo),
    "technocratic <-> regime security":          cross_set_similarities(wv, tech, secu),
    "political governance <-> ideological":      cross_set_similarities(wv, polg, ideo),
    "political governance <-> regime security":  cross_set_similarities(wv, polg, secu),
    "political governance <-> technocratic":     cross_set_similarities(wv, polg, tech),
    "ideological <-> regime security":           cross_set_similarities(wv, ideo, secu),
    "technocratic (within)":                     intra_set_similarities(wv, tech),
    "political governance (within)":             intra_set_similarities(wv, polg),
    "ideological (within)":                      intra_set_similarities(wv, ideo),
    "regime security (within)":                  intra_set_similarities(wv, secu),
}

rows = []
for name, vals in comparisons.items():
    mean, lo, hi = bootstrap_ci(vals)
    rows.append({
        "comparison": name, "n_pairs": len(vals), "mean": round(mean, 4),
        "ci_95": f"[{lo:.3f}, {hi:.3f}]",
        "z_vs_random": round(n_vocab.z(mean), 2),
        "z_vs_freq_matched": round(n_freq.z(mean), 2),
        "above_95th_pct": mean > n_vocab.p95,
    })
pd.DataFrame(rows)

## Permutation tests

Any significant result here means the vocabularies are
distinguishable, albeit a lexical one. Cosine CANNOT show heirarchy. 

In [ ]:
from xi_discourse.stats import permutation_cluster_test

perm_rows = []
for name, (a, b) in {
    "technocratic vs ideological":             (tech, ideo),
    "technocratic vs regime security":         (tech, secu),
    "political governance vs ideological":     (polg, ideo),
    "political governance vs regime security": (polg, secu),
    "ideological vs regime security":          (ideo, secu),
}.items():
    r = permutation_cluster_test(wv, a, b)
    perm_rows.append({
        "split": name, "observed_gap": round(r.observed, 4),
        "null_mean": round(r.null_mean, 4), "p_value": round(r.p_value, 4),
        "reading": "separable" if r.separable else "entangled",
    })
pd.DataFrame(perm_rows)

# Standard dev.

Every headline pair, reported as mean ± SD across the five seeds

In [ ]:
from xi_discourse.stats import seed_spread

pairs = [
    # bare 治理 -- polysemous, dominated by its environmental sense
    ("治理", "安全"), ("治理", "党"), ("治理", "经济"), ("治理", "市场"),
    # political-governance compounds -- the constructs the debate is actually about
    ("国家治理体系", "党"), ("国家治理体系", "国家安全"), ("国家治理体系", "经济"),
    ("治理能力", "党"), ("治理能力", "国家安全"), ("治理能力", "市场"),
    ("依法治国", "党"), ("依法治国", "经济"), ("制度", "党"),
]

rows = []
for a, b in pairs:
    if a not in wv or b not in wv:
        continue
    m, sd, n = seed_spread(models, a, b)
    rows.append({
        "pair": f"{lexicon.EN.get(a, a)} - {lexicon.EN.get(b, b)}",
        "zh": f"{a}-{b}", "mean": round(m, 4), "sd": round(sd, 4),
        "n_seeds": n, "z_vs_random": round(n_vocab.z(m), 2),
    })
pd.DataFrame(rows).sort_values("mean", ascending=False)

## Directional measures of subordination

Cosine is symmetric, so it cannot express hierarchy 

"Semantic axis projection** 
Each administrative term is placed on a single regime-security ↔ economic axis


See Bolukbasi et al. 2016; Kozlowski, Taddy & Evans 2019.

In [ ]:
from xi_discourse.subordination import project_with_ci

projections = project_with_ci(wv, tech + polg, secu, list(lexicon.ECONOMIC_COOPERATIVE))
pd.DataFrame([{
    "term": lexicon.EN.get(p.word, p.word), "zh": p.word,
    "field": "political governance" if p.word in polg else "technocratic",
    "axis_position": round(p.score, 4),
    "ci_95": f"[{p.ci_lo:.3f}, {p.ci_hi:.3f}]",
    "ci_excludes_zero": (p.ci_lo > 0) or (p.ci_hi < 0),
} for p in projections]).sort_values("axis_position", ascending=False)

If governance discourse is 'structurally'
embedded in security discourse, then P(security | governance) >
P(governance | security)

In [ ]:
from xi_discourse.subordination import asymmetry_matrix, mean_asymmetry

admin = tech + polg
asym_sec = asymmetry_matrix(sentences, admin, secu)
asym_ideo = asymmetry_matrix(sentences, admin, ideo)

print(f"mean asymmetry, administrative -> regime security : {mean_asymmetry(asym_sec):+.4f}")
print(f"mean asymmetry, administrative -> ideological     : {mean_asymmetry(asym_ideo):+.4f}")

party = [a for a in asym_ideo if a.b == "党" and np.isfinite(a.lift)]
print(f"\nadministrative terms occurring with 党 more often than the reverse: "
      f"{sum(a.lift > 1 for a in party)}/{len(party)}")
print(f"median lift: {np.median([a.lift for a in party]):.2f}x\n")

display(pd.DataFrame([{
    "administrative": lexicon.EN.get(a.a, a.a), "zh": a.a,
    "P(party|admin)": round(a.p_b_given_a, 4),
    "P(admin|party)": round(a.p_a_given_b, 4),
    "asymmetry": round(a.asymmetry, 4), "lift": f"{a.lift:.2f}x",
} for a in sorted(party, key=lambda a: -a.lift)]))

pd.DataFrame([{
    "administrative": lexicon.EN.get(a.a, a.a), "frame": lexicon.EN.get(a.b, a.b),
    "P(frame|admin)": round(a.p_b_given_a, 4), "P(admin|frame)": round(a.p_a_given_b, 4),
    "asymmetry": round(a.asymmetry, 4), "lift": round(a.lift, 2),
} for a in asym_sec if np.isfinite(a.asymmetry)]
).sort_values("asymmetry", ascending=False).head(12)

What share of a term's nearest neighbours belong to each
discourse field, benchmarked against controls.


Without control a high share means nothing, since large fields dominate the neighbourhoods of common words
regardless of any real relationship.

In [ ]:
from xi_discourse.subordination import neighbourhood_profile

fields = {"technocratic": tech, "political_governance": polg,
          "ideological": ideo, "regime_security": secu, "ecological": ecol}

for word in ("治理", "管理", "政策", "国家治理体系", "治理能力", "依法治国"):
    if word not in wv:
        continue
    p = neighbourhood_profile(wv, word, fields)
    print(f"\n{lexicon.EN.get(word, word)} ({word}) - top-{p.topn} neighbours")
    for f in p.shares:
        print(f"   {f:22s} {p.shares[f]:5.1%}   (control {p.control_shares[f]:5.1%},"
              f" excess {p.excess(f):+.1%})")
    print("   nearest:", "  ".join(f"{w}:{s:.2f}" for w, s in p.neighbours[:8]))

## On ecological security

治理 is polysemous in this corpus (political governance (国家治理, 社会治理) *and*
environmental remediation (环境治理, 水污染治理)).

Here we project the ecological direction out of both vectors.

In [ ]:
from xi_discourse.confounds import ecology_direction, mediation_check, sense_partition

part = sense_partition(sentences)
print("sentence registers:", part.summary(), "\n")

med = mediation_check(wv, [
    ("治理", "安全"), ("治理", "国家安全"), ("管理", "安全"),
    ("治理", "政治"), ("治理", "党"),
    ("国家治理体系", "党"), ("国家治理体系", "国家安全"),
    ("治理能力", "国家安全"), ("依法治国", "党"),
], ecology_direction(wv))

pd.DataFrame([{
    "pair": f"{lexicon.EN.get(m.a, m.a)} - {lexicon.EN.get(m.b, m.b)}",
    "zh": f"{m.a}-{m.b}",
    "raw": round(m.raw, 4), "ecology_residualised": round(m.residual, 4),
    "attenuation": f"{m.attenuation:.1%}", "survives": m.survives,
} for m in med])

## Belt and Road

Are Chinese policy initiatives (in this case the BRI) pragmatic rather than security-driven?

In [ ]:
from xi_discourse.bri import framing_test, neighbour_evidence

tests = framing_test(wv)
display(pd.DataFrame([{
    "term": t.term, "security_framing": round(t.security_score, 4),
    "economic_framing": round(t.economic_score, 4),
    "ideological_framing": round(t.ideological_score, 4),
    "margin (sec - econ)": round(t.margin, 4), "verdict": t.verdict,
} for t in tests if np.isfinite(t.security_score)]))

for term in lexicon.BRI_TERMS:
    ev = neighbour_evidence(wv, term, 12)
    if ev:
        print(f"\n{term} nearest neighbours:")
        print("  " + "  ".join(f"{w}:{s:.2f}" for w, s in ev))

## Conference figures

In [ ]:
from IPython.display import Image, display

from xi_discourse import viz

groups = {"technocratic": tech, "political_governance": polg,
          "ideological": ideo, "regime_security": secu, "ecological": ecol}
all_terms = tech + polg + ideo + secu
all_fields = (["technocratic"] * len(tech) + ["political_governance"] * len(polg)
              + ["ideological"] * len(ideo) + ["regime_security"] * len(secu))

paths = [
    viz.plot_vector_space(wv, groups),
    viz.plot_similarity_matrix(wv, all_terms, all_fields),
    viz.plot_axis_projection(
        projections, "Regime security", "Economic",
        {**{t: "technocratic" for t in tech},
         **{t: "political_governance" for t in polg}}),
    viz.plot_bri_framing(tests),
]
for path in paths:
    print(path)
    display(Image(filename=str(path), width=760))

## Discussion

### What the corpus supports

1. Political-governance vocabulary is party-embedded; generic technocratic vocabulary
is not. Political governance sits at 0.33 from
ideological vocabulary, while technocratic vocabulary sits at 0.21. The strongest single association measured anywhere in the
study is 依法治国 ↔ 党 (law-based governance ↔ party).

2. 16 of 17 administrative terms occur with 党 more often than the reverse, median lift
5.7×, with a clean gradient by how governance-specific the term is

| term | lift | | term | lift |
|---|---|---|---|---|
| 基层治理 grassroots governance | 119× | | 管理 administration | 5.9× |
| 社会治理 social governance | 35× | | 治理 governance | 4.6× |
| 国家治理体系 national governance system | 19× | | 改革 reform | 1.5× |
| 治理能力 governance capacity | 17× | | 创新 innovation | 1.4× |
| 依法治国 law-based governance | 8.9× | | **经济 economy** | **0.91×** |

Over a quarter of sentences containing 依法治国 also contain 党; only 3% of 党 sentences
contain 依法治国

3. Every political-governance term leans securitised. All six
project onto the regime-security side of a security/economic axis with bootstrap CIs
excluding zero, while every technocratic-economic term projects to the economic side.

### What the corpus does not support

1. The bare term 治理 does not mean all that much 66% of its nearest neighbours are
ecological against a 5% frequency-matched baseline. Any claim resting on 治理 alone is a claim about river and pollution management.

2. The strong proximity claim fails. Every field pair remains statistically separable
(p ≈ 0), and every cross-field cosine sits at or near the random-pair baseline. Vocabularies
that are lexically distinguishable can still stand in a hierarchical relation but "everything collapses into one ideological
cluster" is not shown here